# WHAM extract -> normalized SMPL `.npz`

Runs [WHAM](https://github.com/yohanshin/WHAM) (world-grounded SMPL) on one monocular
badminton clip and writes `<video_id>.wham.npz`, which `tools/smpl_to_skeleton.py
--wham-output` turns into `skeleton.json v2`.

- Plan: `docs/superpowers/specs/2026-07-23-monocular-smpl-skeleton-design.md`
- Spec / output contract: see `tools/colab/README.md` in this same folder.

Requires Colab Runtime -> GPU. See the README for the one-time SMPL model setup.


In [ ]:
!git clone https://github.com/yohanshin/WHAM.git
%cd WHAM
!bash install.sh          # WHAM's own setup (torch, deps, checkpoints)
!pip install smplx


In [ ]:
from google.colab import files
print("Upload SMPL_NEUTRAL.pkl (from smpl.is.tue.mpg.de)")
up = files.upload()
import os, shutil
os.makedirs("body_models/smpl", exist_ok=True)
shutil.copy(next(iter(up)), "body_models/smpl/SMPL_NEUTRAL.pkl")


In [ ]:
VIDEO_ID = "test_N"
up = files.upload()                     # the .mp4
clip = next(iter(up))
!python demo.py --video "{clip}" --output_pth output/{VIDEO_ID} --save_pkl --visualize


In [ ]:
import numpy as np, torch, joblib, glob, smplx

res = joblib.load(sorted(glob.glob(f"output/{VIDEO_ID}/*.pkl"))[0])
track = res[sorted(res.keys())[0]]     # first tracked person

# WHAM stores world-grounded params when available.
pose  = np.asarray(track.get("pose_world", track["pose"]), dtype=np.float32)   # (T,72)
transl = np.asarray(track.get("trans_world", track["trans"]), dtype=np.float32) # (T,3)
betas = np.asarray(track["betas"], dtype=np.float32)
if betas.ndim == 2:
    betas = betas.mean(0)              # (10,)

body = smplx.create("body_models", model_type="smpl", gender="neutral", batch_size=pose.shape[0])
out = body(global_orient=torch.tensor(pose[:, :3]),
           body_pose=torch.tensor(pose[:, 3:72]),
           betas=torch.tensor(betas[None].repeat(pose.shape[0], 0)),
           transl=torch.tensor(transl))
joints3d = out.joints.detach().cpu().numpy()[:, :24, :]   # (T,24,3)

# fps from the source video
import cv2
fps = cv2.VideoCapture(clip).get(cv2.CAP_PROP_FPS) or 30.0

np.savez(f"{VIDEO_ID}.wham.npz", joints3d=joints3d, pose=pose,
         betas=betas, transl=transl, fps=np.array(fps))
print("shapes:", joints3d.shape, pose.shape, betas.shape, transl.shape, "fps", fps)
files.download(f"{VIDEO_ID}.wham.npz")
